# ORCA helper (ready to run, nothing to upload for Part A)
**Part A:** `Runtime -> Run all` up to the end of the 3rd code cell. It uses the built-in 102-solvent table and downloads `orca_inputs.zip` (inputs + `names.csv`).
**Run ORCA yourself** on your computer (it cannot run in Colab): unzip, then `bash run_orca_all.sh`.
**Part B (only AFTER ORCA has finished):** run the last cell and upload the `.out` files + `names.csv`. It downloads `qm_descriptors.csv` for the main notebook. Do NOT click Run all before ORCA is done; stop after Part A.

In [ ]:
!pip -q install rdkit

In [ ]:
import pandas as pd, numpy as np, re, zipfile, io
from pathlib import Path
from rdkit import Chem
from rdkit.Chem import AllChem
from google.colab import files
import io
SMILES_CSV = r'''
name,smiles
DMF,CN(C)C=O
NMP,CN1CCCC1=O
DMAc,CC(=O)N(C)C
DMSO,CS(C)=O
Acetonitrile,CC#N
Acetone,CC(C)=O
Sulfolane,O=S1(=O)CCCC1
THF,C1CCOC1
Ethyl acetate,CCOC(C)=O
Dimethyl carbonate,COC(=O)OC
Toluene,Cc1ccccc1
Ethanol,CCO
Methanol,CO
Dichloromethane,ClCCl
Dioxane,C1COCCO1
Anisole,COc1ccccc1
Cyclohexanone,O=C1CCCCC1
2-Butanone,CCC(C)=O
Pyridine,c1ccncc1
gamma-Butyrolactone,O=C1CCCO1
Cyrene,C1CC(=O)C2COC1O2
GVL,CC1CCC(=O)O1
Propylene carbonate,CC1COC(=O)O1
2-MeTHF,CC1CCCO1
Ethyl lactate,CC(O)C(=O)OCC
Dimethyl isosorbide,COC1COC2C1OCC2OC
Glycerol carbonate,OCC1COC(=O)O1
TMO,CC1(C)CCC(C)(C)O1
CPME,COC1CCCC1
NBP,CCCCN1CCCC1=O
PolarClean,COC(=O)C(C)CCC(=O)N(C)C
Methyl levulinate,COC(=O)CCC(C)=O
Ethyl levulinate,CCOC(=O)CCC(C)=O
Furfural,O=Cc1ccco1
Dimethyl succinate,COC(=O)CCC(=O)OC
Diethyl carbonate,CCOC(=O)OCC
Triacetin,CC(=O)OCC(COC(C)=O)OC(C)=O
Butyl lactate,CCCCOC(=O)C(C)O
Tetrahydrofurfuryl alcohol,OCC1CCCO1
Dimethyllactamide,CC(O)C(=O)N(C)C
Limonene,CC1=CCC(CC1)C(C)=C
p-Cymene,Cc1ccc(C(C)C)cc1
Dioxolane,C1COCO1
Formamide,NC=O
N-Methylformamide,CNC=O
"N,N-Dimethylpropionamide",CCC(=O)N(C)C
Tetramethylurea,CN(C)C(=O)N(C)C
DMPU,CN1CCCN(C)C1=O
"DMI (1,3-dimethylimidazolidinone)",CN1CCN(C)C1=O
HMPA,CN(C)P(=O)(N(C)C)N(C)C
2-Pyrrolidone,O=C1CCCN1
NEP,CCN1CCCC1=O
NOP,CCCCCCCCN1CCCC1=O
N-Formylmorpholine,O=CN1CCOCC1
Nitromethane,C[N+](=O)[O-]
Nitrobenzene,[O-][N+](=O)c1ccccc1
Propionitrile,CCC#N
Benzonitrile,N#Cc1ccccc1
Ethylene carbonate,O=C1OCCO1
Acetophenone,CC(=O)c1ccccc1
Cyclopentanone,O=C1CCCC1
MIBK,CC(C)CC(C)=O
Diacetone alcohol,CC(=O)CC(C)(C)O
Diethyl ether,CCOCC
MTBE,COC(C)(C)C
"1,2-Dimethoxyethane",COCCOC
Diglyme,COCCOCCOC
Methyl acetate,COC(C)=O
Butyl acetate,CCCCOC(C)=O
Isopropyl acetate,CC(C)OC(C)=O
Hexane,CCCCCC
Heptane,CCCCCCC
Cyclohexane,C1CCCCC1
Benzene,c1ccccc1
p-Xylene,Cc1ccc(C)cc1
Chloroform,ClC(Cl)Cl
"1,2-Dichloroethane",ClCCCl
Chlorobenzene,Clc1ccccc1
Isopropanol,CC(C)O
1-Propanol,CCCO
1-Butanol,CCCCO
tert-Butanol,CC(C)(C)O
Ethylene glycol,OCCO
Propylene glycol,CC(O)CO
"1,3-Propanediol",OCCCO
Glycerol,OCC(O)CO
2-Methoxyethanol,COCCO
Diethylene glycol,OCCOCCO
Water,O
Acetic acid,CC(=O)O
Triethylamine,CCN(CC)CC
Methyl lactate,COC(=O)C(C)O
Levulinic acid,CC(=O)CCC(=O)O
Diethyl succinate,CCOC(=O)CCC(=O)OCC
Dimethyl glutarate,COC(=O)CCCC(=O)OC
Dimethyl adipate,COC(=O)CCCCC(=O)OC
Dimethyl 2-methylglutarate,COC(=O)CCC(C)C(=O)OC
Furfuryl alcohol,OCc1ccco1
2-Methylfuran,Cc1ccco1
"2,5-Dimethylfuran",Cc1ccc(C)o1
"1,8-Cineole",CC1(C2CCC(O1)(CC2)C)C
alpha-Pinene,CC1=CCC2CC1C2(C)C
'''
df = pd.read_csv(io.StringIO(SMILES_CSV.strip()))  # built-in 102-solvent table, nothing to upload
# choose which solvents: set NAMES=None for all rows, or a list of names
NAMES = None
METHOD, NPROCS, MAXCORE, N_CONFS = "r2SCAN-3c", 4, 3000, 30
if NAMES is not None: df = df[df.name.isin(NAMES)]
print(len(df), "solvents")

In [ ]:
import shutil; shutil.rmtree("orca_inputs", ignore_errors=True)
out = Path("orca_inputs"); out.mkdir(exist_ok=True); ok, bad = [], []
for n, smi in zip(df.name, df.smiles):
    try:
        mol = Chem.AddHs(Chem.MolFromSmiles(smi))
        cids = list(AllChem.EmbedMultipleConfs(mol, numConfs=N_CONFS, randomSeed=42))
        res = AllChem.MMFFOptimizeMoleculeConfs(mol, maxIters=5000)
        best = cids[int(np.argmin([r[1] for r in res]))]
        slug = "".join(c if c.isalnum() else "_" for c in n)
        (out/f"{slug}.xyz").write_text(Chem.MolToXYZBlock(mol, confId=best))
        (out/f"{slug}.inp").write_text(f"! {METHOD} Opt Freq TightSCF\n%pal nprocs {NPROCS} end\n%maxcore {MAXCORE}\n* xyzfile 0 1 {slug}.xyz\n")
        pd.DataFrame({"slug":[slug],"name":[n]}).to_csv(out/"names.csv", mode="a", header=not (out/"names.csv").exists(), index=False)
        ok.append(slug)
    except Exception as e:
        bad.append((n, str(e)[:60]))
print("written:", len(ok), "| failed:", bad)
with zipfile.ZipFile("orca_inputs.zip","w") as z:
    for f in out.iterdir(): z.write(f, f.name)
files.download("orca_inputs.zip")
print("Run on your machine, e.g.:  for f in *.inp; do orca $f > ${f%.inp}.out; done   (check frequencies: no imaginary ones)")

## Part B: parse finished ORCA outputs

In [ ]:
print("Upload the .out files (several at once) and names.csv from orca_inputs")
up = files.upload()
names = pd.read_csv(io.StringIO(up["names.csv"].decode())).set_index("slug")["name"].to_dict() if "names.csv" in up else {}
def parse_orca(txt):
    d = {}
    d["converged"] = ("THE OPTIMIZATION HAS CONVERGED" in txt) or ("OPTIMIZATION RUN DONE" in txt)
    d["normal_termination"] = "ORCA TERMINATED NORMALLY" in txt
    d["imaginary_freqs"] = len(re.findall(r"\*\*\*imaginary mode\*\*\*", txt)) 
    m = re.findall(r"Magnitude \(Debye\)\s*:\s*([-\d.]+)", txt); d["dipole_D"] = float(m[-1]) if m else np.nan
    m = re.findall(r"FINAL SINGLE POINT ENERGY\s+([-\d.]+)", txt); d["E_h"] = float(m[-1]) if m else np.nan
    blocks = txt.split("ORBITAL ENERGIES")
    homo = lumo = np.nan
    if len(blocks) > 1:
        occ = []
        for line in blocks[-1].split("\n")[3:]:
            p = line.split()
            if len(p) >= 4 and p[0].isdigit():
                occ.append((float(p[1]), float(p[3])))
            elif occ: break
        occupied = [e for o, e in occ if o > 0]; virt = [e for o, e in occ if o == 0]
        if occupied and virt: homo, lumo = max(occupied), min(virt)
    d["homo_eV"], d["lumo_eV"] = homo, lumo; d["gap_eV"] = lumo - homo
    return d
rows = []
for fn, b in up.items():
    if fn.endswith(".out"):
        slug = fn[:-4]; r = parse_orca(b.decode(errors="ignore")); r["name"] = names.get(slug, slug); rows.append(r)
assert rows, 'No .out files were uploaded. Run ORCA first, then upload the .out files (and names.csv) here.'
Q = pd.DataFrame(rows); display(Q)
bad = Q[(~Q.normal_termination) | (Q.imaginary_freqs > 0) | Q.dipole_D.isna()]
if len(bad): print("CHECK THESE (failed / imaginary frequencies / no dipole):", bad.name.tolist())
Q[["name","dipole_D","E_h","homo_eV","lumo_eV","gap_eV"]].to_csv("qm_descriptors.csv", index=False)
files.download("qm_descriptors.csv")